# Use Case 2: Health & Life Science — Molecular VQE & Classical Shadows

This notebook estimates the molecular ground state energy of small molecules (e.g. $H_2$) using a hybrid quantum-classical workflow:
1. **Classical Stage:** Hartree-Fock and AVAS active space selection (PySCF) to build a Jordan-Wigner qubit Hamiltonian.
2. **Quantum VQE:** Variational parameter optimization using an EfficientSU2 ansatz and SPSA optimizer on simulator or real VLQ hardware.
3. **Post-Processing:** Classical Shadows observable reconstruction protocol for sample-efficient energy estimation.

- **Target Backend Options:** Local Simulator (`SIM`) or Real QPU (`VLQ` via LEXIS)
- **Environment:** Pre-loaded via LUMI Lmod module (`module load Lumi-IQ-qaas/VLQ`)

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# Add code directory to path
sys.path.insert(0, os.path.abspath("code"))
sys.path.insert(0, os.path.abspath("../.."))

import numpy as np
import matplotlib.pyplot as plt
from main import run_workflow, get_backend
print("Environment initialized successfully!")


### 1. Classical Stage: Molecule and Qubit Hamiltonian
Define the molecular geometry and compute the electronic Hamiltonian using PySCF and Jordan-Wigner transformation.

In [ ]:
from hamiltonian import run_hf, select_active_space, active_space_hamiltonian, qubit_hamiltonian

atom = "H 0 0 0; H 0 0 0.74"
basis = "sto-3g"

mf = run_hf(atom, basis, spin=0)
n_orbitals, n_alpha, n_beta = select_active_space(mf, ao_labels=["H 1s"], threshold=0.2)
fermionic_ham = active_space_hamiltonian(mf, n_alpha, n_beta, n_orbitals)
hamiltonian_mat, qubit_op = qubit_hamiltonian(fermionic_ham)

print(f"Active space: {n_orbitals} orbitals, {n_alpha} alpha, {n_beta} beta electrons.")
print(f"Qubit Hamiltonian generated: {qubit_op.num_qubits} qubits, {len(qubit_op)} Pauli terms.")
print(f"Hartree-Fock energy: {mf.e_tot:.6f} Ha (Nuclear repulsion: {mf.mol.energy_nuc():.6f} Ha)")


### 2. Hybrid VQE & Classical Shadows Execution
Select `--backend SIM` for fast local testing or `--backend VLQ` to dispatch to the real 24-qubit star-topology QPU via LEXIS.

In [ ]:
# Set target backend: "SIM" (fast local Aer simulator) or "VLQ" (real QPU via LEXIS)
TARGET_BACKEND = "VLQ"
MAX_ITERS = 15
SHOTS = 256

print(f"Executing hybrid workflow on {TARGET_BACKEND}...")
energy = run_workflow(
    atom=atom,
    basis=basis,
    spin=0,
    ao_labels=["H 1s"],
    threshold=0.2,
    backend=TARGET_BACKEND,
    optimization_level=1,
    max_iters=MAX_ITERS,
    shots=SHOTS,
    trajectory_file="vqe_trajectory.csv",
    replay_shots=SHOTS,
    N=100,
    K=10,
    max_circuits_per_job=50,
    report_per_job_progress=True,
)
print(f"Workflow complete. Final ground state energy: {energy:.6f} Ha")


### 3. Energy Convergence Visualization
Plot the VQE optimization trajectory against the exact Full Configuration Interaction (FCI) target energy.

In [ ]:
import pandas as pd

df = pd.read_csv("vqe_trajectory.csv")
col = "noisy_energy" if "noisy_energy" in df.columns else df.columns[1]
energies = df[col].dropna().tolist()
iters = list(range(1, len(energies) + 1))
E_CI = -1.85238817356958

plt.figure(figsize=(9, 5))
plt.plot(iters, energies, marker="o", markersize=4, linewidth=2, color="#1f77b4", label="VQE SPSA Energy")
plt.axhline(y=E_CI, color="#2ca02c", linestyle="--", linewidth=2, label=f"Full CI Target ({E_CI:.4f} Ha)")
plt.title("VQE Energy Convergence vs Target Energy", fontsize=13, fontweight="bold")
plt.xlabel("Optimizer Iteration", fontsize=11)
plt.ylabel("Energy (Hartree)", fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("result_health.png", dpi=150)
plt.show()
